# PatchCore Defect Detector — Colab 執行

在 Colab 跑 `patchcore/` 套件。**先到 `Runtime → Change runtime type → GPU`** 開 GPU(Colab 是 CUDA,程式會自動偵測使用)。
逐格由上往下執行即可。

## 1. 取得程式碼(clone 或更新)

In [ ]:
import os
REPO = "/content/Unsupervised-Learning-Defect-Detector"
if not os.path.exists(REPO):
    !git clone https://github.com/njzneverdie/Unsupervised-Learning-Defect-Detector.git {REPO}
else:
    !cd {REPO} && git pull

## 2. 檢查 GPU 並安裝相依套件
Colab 已內建 torch / torchvision / matplotlib;只補裝 opencv 與 sklearn。

In [ ]:
!nvidia-smi -L || echo ">>> 沒偵測到 GPU:請到 Runtime > Change runtime type > GPU"
!pip -q install opencv-python-headless scikit-learn

## 3. 下載資料集(Kaggle)
執行後上傳你的 `kaggle.json`(Kaggle → Settings → API → Create New Token)。

In [ ]:
import os
kaggle_json_path = "/root/.kaggle/kaggle.json"
if not os.path.exists(kaggle_json_path):
    from google.colab import files
    print("請上傳你的 kaggle.json")
    up = files.upload()
    os.makedirs("/root/.kaggle", exist_ok=True)
    with open(kaggle_json_path, "wb") as f:
        f.write(up["kaggle.json"])
    os.chmod(kaggle_json_path, 0o600)
print("Kaggle credentials ready.")

In [ ]:
!pip -q install kaggle
!kaggle datasets download -d dragonzhang/grainsize-train -p /content
!unzip -q -o /content/grainsize-train.zip -d /content/grainsize_dataset
!echo "影像數:" && find /content/grainsize_dataset -type f | wc -l

## 4. ① 建記憶庫

先用 `--limit 100` 跑通(只用 100 張正常圖建庫)。要更完整可調大或拿掉 `--limit`。
記憶庫只能用**正常(defect-free)**影像;第一版先假設整批≈正常。

In [ ]:
%cd /content/Unsupervised-Learning-Defect-Detector/patchcore
!python build_memory.py --data /content/grainsize_dataset --out memory.pt --limit 100

## 5. ③ 量化評估(合成缺陷 → ROC-AUC / IoU)

In [ ]:
!python evaluate.py --data /content/grainsize_dataset --memory memory.pt

## 6. ② 對單張影像出 heatmap

In [ ]:
import glob
from IPython.display import Image, display
imgs = sorted(glob.glob('/content/grainsize_dataset/**/*.*', recursive=True))
imgs = [p for p in imgs if p.lower().endswith(('.jpg','.jpeg','.png','.tif','.tiff'))]
target = imgs[0]
print("偵測:", target)
!python detect.py --image "{target}" --memory memory.pt --out result.png --quantile 0.95
display(Image('result.png'))

## 調參
所有參數在 `patchcore/config.py`:`tile_size`/`input_size`(小缺陷解析度)、
`presample_max`/`coreset_ratio`(記憶庫大小↔速度)、`gaussian_sigma`(平滑)、`global_quantile`(門檻)。
改完重跑第 4~6 格即可。